# K09_00 - Iris: Erstes Keras-Modell

## Lernziele
Nach diesem Notebook koennen Sie:

- ein erstes Klassifikationsmodell mit **Keras** aufbauen
- **One-Hot-Encoding** fuer Mehrklassenklassifikation einordnen
- Daten in Trainings-, Validierungs- und Testdaten aufteilen
- Eingabemerkmale skalieren
- ein `Sequential`-Modell definieren, kompilieren, trainieren und evaluieren
- die Begriffe **Epoch** und **Batch** erklaeren
- Lernkurven und Konfusionsmatrix interpretieren


## Didaktischer Fokus

Dieses Notebook ist der **Einstieg in Keras**.

Wir setzen den vollstaendigen Workflow direkt um:
1. Iris-Daten laden
2. Zielvariable One-Hot-kodieren
3. `train_test_split` mit `stratify`
4. Merkmale skalieren
5. Modell definieren und kompilieren
6. Modell trainieren (mit explizitem Validierungsset)
7. Vorhersagen und Evaluation
8. Lernkurven visualisieren

**Hinweis zur Keras-Version:**  
Dieses Notebook ist kompatibel mit TensorFlow >= 2.12.  
In Google Colab ist TensorFlow normalerweise bereits vorinstalliert.


## Imports und Reproduzierbarkeit


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seeds erhoehen die Reproduzierbarkeit, garantieren aber je nach
# TensorFlow-Version / Hardware nicht immer bitgenau identische Ergebnisse.
np.random.seed(42)

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

import tensorflow as tf
tf.random.set_seed(42)

from tensorflow import keras
from tensorflow.keras import layers

print('Alle Bibliotheken geladen.')
print(f'TensorFlow-Version: {tf.__version__}')


## 1. Iris-Daten laden


In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names  = iris.target_names

print('Shape von X:', X.shape)
print('Shape von y:', y.shape)
print('Klassen:    ', target_names)

unique, counts = np.unique(y, return_counts=True)
class_counts = pd.DataFrame({'Klasse': target_names[unique], 'Anzahl': counts})
print('\nKlassenverteilung:')
print(class_counts)


**Mini-Uebung 1:** Wie viele Samples und Features hat der Datensatz?  
Welche Klassen gibt es? Wie viele Samples gehoeren zu jeder Klasse?


## 2. Zielvariable One-Hot-kodieren

Fuer drei Klassen erzeugen wir Zielvektoren:
- Klasse 0 -> `[1, 0, 0]`
- Klasse 1 -> `[0, 1, 0]`
- Klasse 2 -> `[0, 0, 1]`

**Warum?** Die Ausgabeschicht hat drei Neuronen (eines pro Klasse).  
Damit die Loss-Funktion `categorical_crossentropy` korrekt funktioniert,
muss der Zielvektor dieselbe Form haben wie die Ausgabe des Modells.

**Hinweis:**  
Alternativ koennte man die Klassenlabels auch als Integer (`0`, `1`, `2`) belassen  
und die Verlustfunktion `sparse_categorical_crossentropy` verwenden.  
In diesem Notebook nutzen wir bewusst One-Hot-Encoding, weil die Softmax-Ausgabe  
so besonders anschaulich interpretiert werden kann.


In [ ]:
y_cat = keras.utils.to_categorical(y, num_classes=3)

print('Shape von y_cat:', y_cat.shape)
print('Erstes Label  (Integer):', y[0],  '-> One-Hot:', y_cat[0])
print('Letztes Label (Integer):', y[-1], '-> One-Hot:', y_cat[-1])


**Mini-Uebung 2:** Was waere der One-Hot-Vektor fuer Klasse 1 (`versicolor`)?  
Warum verwendet man One-Hot-Encoding anstatt einfach `0`, `1`, `2` als Zielwerte zu behalten?


## 3. Trainings-, Validierungs- und Testdaten erzeugen

Wichtig: `stratify=y` verwendet das **originale** Integer-Label `y` (nicht `y_cat`),  
damit die Klassenverteilung in Train und Test aehnlich bleibt.

Fuer den **Validierungsanteil** waehrend des Trainings splitten wir zusaetzlich  
aus den Trainingsdaten - ebenfalls stratifiziert.

Dadurch entstehen hier ungefaehr:
- 64 % Trainingsdaten
- 16 % Validierungsdaten
- 20 % Testdaten


In [ ]:
# Haupt-Split: 80 % Train, 20 % Test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)

# Validierungsset aus den Trainingsdaten (stratifiziert)
y_train_full_int = np.argmax(y_train_full, axis=1)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42, stratify=y_train_full_int
)

print('Trainingsdaten:   ', X_train.shape)
print('Validierungsdaten:', X_val.shape)
print('Testdaten:        ', X_test.shape)


**Mini-Uebung 3:** Warum teilen wir in drei Mengen auf (Train / Validation / Test)  
und nicht nur in zwei (Train / Test)?


## 4. Eingaben skalieren

Neuronale Netze sind empfindlich gegenueber unterschiedlichen Groessenordnungen.  
Der `StandardScaler` verschiebt jedes Merkmal auf Mittelwert ~ 0 und Standardabweichung ~ 1.

**Wichtig:** `fit_transform` nur auf Trainingsdaten, `transform` auf Validierung und Test.

**Hinweis zum Unterschied zu bisherigen Notebooks:**  
In Kapitel 7 und 8 haben wir den Scaler in eine `sklearn`-Pipeline eingebettet.  
Keras-Modelle lassen sich nicht direkt in sklearn-Pipelines einfuegen (ohne Wrapper).  
Deshalb skalieren wir hier manuell - das Prinzip bleibt dasselbe:  
`fit` ausschliesslich auf Trainingsdaten, `transform` auf Validierung und Test.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform nur auf Trainingsdaten
X_val_scaled   = scaler.transform(X_val)          # nur transform - kein fit!
X_test_scaled  = scaler.transform(X_test)         # nur transform - kein fit!

print('Mittelwert (Train, nach Skalierung):', X_train_scaled.mean(axis=0).round(4))
print('Std       (Train, nach Skalierung):', X_train_scaled.std(axis=0).round(4))


## 5. Modell definieren und kompilieren

Wir bauen ein kleines `Sequential`-Modell:

```
Input (4 Merkmale)
    |
Dense(16, relu)    <- Hidden Layer
    |
Dense(3, softmax)  <- Ausgabeschicht (eine Klasse pro Neuron)
```

- **`relu`**: Aktivierungsfunktion im Hidden Layer - setzt negative Werte auf 0
- **`softmax`**: Ausgaben koennen als Klassenwahrscheinlichkeiten interpretiert werden; die Summe betraegt 1
- **`categorical_crossentropy`**: passende Loss-Funktion fuer One-Hot-Ziele
- **`adam`**: adaptiver Gradient-Optimierer - passt die Lernrate automatisch an und ist ein guter Standard fuer den Einstieg


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(3,  activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


**Was bedeutet `model.summary()`?**

- **Param #** zeigt die Anzahl lernbarer Gewichte pro Layer.
- Hidden Layer Dense(16): 4 Eingaben x 16 Neuronen + 16 Bias = **80 Parameter**
- Ausgabe-Layer Dense(3): 16 Eingaben x 3 Neuronen + 3 Bias = **51 Parameter**
- Gesamt: **131 trainierbare Parameter** - sehr klein fuer einen Einstieg.


## 6. Modell trainieren

### Zwei zentrale neue Begriffe: Epoch und Batch

**Epoch:**  
Ein vollstaendiger Durchlauf durch alle Trainingsdaten.  
Nach `epochs=30` hat das Modell die Daten **30-mal gesehen**.  
Pro Epoch werden die Gewichte des Modells aktualisiert.

**Batch:**  
Statt alle Daten auf einmal zu verarbeiten, teilt das Modell sie in kleine Pakete auf.  
`batch_size=16` bedeutet: das Modell verarbeitet **16 Beispiele gleichzeitig**  
und aktualisiert danach die Gewichte.

```
Trainingsdaten: 96 Beispiele, batch_size=16

Epoch 1:
  Batch 1: Beispiele  1-16  -> Gewichte aktualisieren
  Batch 2: Beispiele 17-32  -> Gewichte aktualisieren
  Batch 3: Beispiele 33-48  -> Gewichte aktualisieren
  Batch 4: Beispiele 49-64  -> Gewichte aktualisieren
  Batch 5: Beispiele 65-80  -> Gewichte aktualisieren
  Batch 6: Beispiele 81-96  -> Gewichte aktualisieren
Epoch 2: (alle Batches erneut, neu gemischt)
...
```

**`verbose=2`** gibt pro Epoch eine Zeile mit Loss und Accuracy aus -  
uebersichtlicher als `verbose=1` (Fortschrittsbalken pro Batch).

Wir uebergeben ausserdem explizit das Validierungsset (`validation_data`).  
So sehen wir in jeder Epoch, wie das Modell auf **unbekannten Daten** abschneidet.


In [ ]:
history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_val_scaled, y_val),
    verbose=2       # eine Zeile pro Epoch - uebersichtlicher als verbose=1
)


## 7. Lernkurven visualisieren

Die Lernkurven zeigen, wie sich **Loss** und **Accuracy** ueber die Epochen entwickeln.  
Ein grosser Abstand zwischen Train und Val deutet auf **Overfitting** hin.

**Hinweis:**  
Bei kleinen Datensaetzen wie Iris koennen die Validierungswerte von Epoche zu Epoche  
leicht schwanken. Das ist normal und kein Hinweis auf einen Fehler.


In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df['loss'],     label='Train Loss')
axes[0].plot(hist_df['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Lernkurve: Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(hist_df['accuracy'],     label='Train Accuracy')
axes[1].plot(hist_df['val_accuracy'], label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Lernkurve: Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


## 8. Vorhersage und Evaluation

Ein trainiertes Keras-Modell kann auf zwei Arten ausgewertet werden:

| Methode | Wann verwenden? |
|---|---|
| `model.evaluate()` | schnell, effizient, gibt Loss und Accuracy zurueck |
| `model.predict()` + sklearn-Metriken | wenn man F1, Precision, Recall oder Konfusionsmatrix braucht |

Beide Methoden muessen dieselbe Accuracy liefern - wenn nicht, liegt ein Fehler im Code vor.  
Wir zeigen hier beide Wege, damit Sie den Zusammenhang sehen.


In [ ]:
# Keras-eigene Evaluation (Loss + Accuracy in einem Schritt)
test_loss, test_acc_keras = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f'Keras evaluate()  -> Loss: {test_loss:.4f} | Accuracy: {test_acc_keras:.4f}')

# Vorhersagen fuer sklearn-Metriken und Konfusionsmatrix
y_prob = model.predict(X_test_scaled, verbose=0)  # Wahrscheinlichkeiten (Softmax-Ausgabe)
y_pred = np.argmax(y_prob, axis=1)                # Klasse mit hoechster Wahrscheinlichkeit
y_true = np.argmax(y_test, axis=1)                # Wahre Klassen (aus One-Hot zurueck)

acc_sklearn = accuracy_score(y_true, y_pred)
print(f'sklearn accuracy_score -> Accuracy: {acc_sklearn:.4f}')
print(f'\nBeide Werte sollten identisch sein: {abs(test_acc_keras - acc_sklearn) < 1e-4}')

# Beispiel: Softmax-Wahrscheinlichkeiten fuer die ersten 5 Testbeispiele
print('\nSoftmax-Wahrscheinlichkeiten (erste 5 Testbeispiele):')
print(pd.DataFrame(y_prob[:5].round(3), columns=target_names))
print('\nSumme jeder Zeile (muss 1.0 sein):', y_prob[:5].sum(axis=1).round(4))


## 9. Konfusionsmatrix


In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(colorbar=False)
plt.title('Konfusionsmatrix - Testdaten')
plt.show()


**Mini-Uebung 4:** Interpretieren Sie die Konfusionsmatrix.  
- Welche Klassen werden zuverlaessig erkannt?
- Gibt es Verwechslungen? Wenn ja: welche Klassen, und warum koennte das so sein?


## Fazit

- Keras ermoeglicht einen sehr klaren Einstieg in neuronale Netze.
- Fuer Mehrklassenklassifikation passen `softmax` und `categorical_crossentropy` zusammen.
- Eine **Epoch** ist ein vollstaendiger Durchlauf durch alle Trainingsdaten.
- Ein **Batch** ist ein Paket von Beispielen, nach dem die Gewichte aktualisiert werden.
- Skalierung und sauberer Train/Val/Test-Split sind auch bei neuronalen Netzen wichtig.
- Lernkurven machen den Trainingsprozess sichtbar.
- `model.evaluate()` und `model.predict()` erganzen sich: gleiche Accuracy, unterschiedliche Ausgabeform.


## 10. Uebungsaufgaben

### Aufgabe 1 - Begriffe erklaeren
Erklaeren Sie in eigenen Worten:
- `Sequential`
- `Dense`
- `softmax`
- `categorical_crossentropy`
- **Epoch**
- **Batch**

### Aufgabe 2 - Begruenden
Warum verwenden wir in diesem Notebook:
- One-Hot-Encoding?
- Skalierung?
- Train / Validation / Test-Split?

### Aufgabe 3 - Modell variieren
Veraendern Sie das Modell und beobachten Sie die Auswirkungen auf die Lernkurven.  
Testen Sie **mindestens drei Varianten**:
- Aendern Sie die Neuronenzahl im Hidden Layer (z. B. 8, 16, 32)
- Aendern Sie die Anzahl der Epochen
- Aendern Sie die Batchgroesse

Vergleichen Sie die Lernkurven der Varianten mit dem Basismodell.


In [ ]:
# Aufgabe 3 - Modell variieren
# Veraendern Sie die kommentierten Parameter und fuehren Sie die Zelle erneut aus.

modell_variante = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(32, activation='relu'),   # <- hier variieren: 8, 16, 32, 64
    layers.Dense(3,  activation='softmax')
])

modell_variante.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_variante = modell_variante.fit(
    X_train_scaled, y_train,
    epochs=50,           # <- hier variieren
    batch_size=8,        # <- hier variieren
    validation_data=(X_val_scaled, y_val),
    verbose=0
)

# Lernkurven der Variante
hist_v = pd.DataFrame(history_variante.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(hist_df['loss'],         label='Basis - Train',   linestyle='--', color='steelblue')
axes[0].plot(hist_df['val_loss'],     label='Basis - Val',     linestyle='--', color='orange')
axes[0].plot(hist_v['loss'],          label='Variante - Train', color='steelblue')
axes[0].plot(hist_v['val_loss'],      label='Variante - Val',   color='orange')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Vergleich Lernkurven: Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True)

# Accuracy
axes[1].plot(hist_df['accuracy'],     label='Basis - Train',   linestyle='--', color='steelblue')
axes[1].plot(hist_df['val_accuracy'], label='Basis - Val',     linestyle='--', color='orange')
axes[1].plot(hist_v['accuracy'],      label='Variante - Train', color='steelblue')
axes[1].plot(hist_v['val_accuracy'],  label='Variante - Val',   color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Vergleich Lernkurven: Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.suptitle('Basismodell (gestrichelt) vs. Variante (durchgezogen)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Finale Accuracy
y_pred_v = np.argmax(modell_variante.predict(X_test_scaled, verbose=0), axis=1)
print(f'Test-Accuracy Basismodell: {accuracy_score(y_true, y_pred):.4f}')
print(f'Test-Accuracy Variante:    {accuracy_score(y_true, y_pred_v):.4f}')
